In [1]:
import random
import numpy as np

In [2]:
class Individual:
    def __init__(self, num_resources, costs, facility_cost):
        self.code = [random.random() < 0.25 for _ in range(num_resources)]
        self.fitness = self.calc_fitness(costs, facility_cost)
        
    def calc_fitness(self, costs, facility_cost):
        if sum(self.code) == 0:
            return float('inf')
        total_cost = self.code @ facility_cost
        built = np.where(self.code)[0]
        total_cost += costs[:, built].min(axis = 1).sum()
        
        return total_cost

In [3]:
# Turnirska selekcija: pravimo turnir nasumično odabranog podskupa jedinki iz naše populacije. Pobednici turnira 
# (najbolje prilagođene jedinke) dobijaju priliku da ostave potomstvo u narednoj generaciji

def tournament_selection(population, tournament_size):
    chosen = random.sample(population, tournament_size) 
    
    # iako je niz population sortiran, sample ne mora da vrati vrednosti u sortiranom redosledu
    chosen = sorted(chosen, key = lambda x: x.fitness)
    
    return chosen[0], chosen[1]

In [4]:
def crossover(parent1, parent2):
    child1 = deepcopy(parent1)
    child2 = deepcopy(parent2)
    
    # uniformno ukrštanje:
    # verovatnoća da dete uzme vrednost na poziciji i od prvog roditelja jednaka je verovatnoći da uzme vrednost 
    # na toj poziciji od drugog roditelja (po 50%)
    for i in range(len(child1.code)):
        # pošto su deca na početku napravljena kao kopije po jednog od roditelja, nema potrebe da obrađujemo
        # slučajeve: ranodm.random() < 0.5 i ranodm.random() > 0.5, već je dovoljan jedan 
        if random.random() > 0.5:
            child1.code[i] = parent2.code[i]
            child2.code[i] = parent1.code[i]
    return child1, child2

In [5]:
def mutation(child, mutation_prob):
    for i in range(len(child.code)):
        if random.random() < mutation_prob:
            child.code[i] = not child.code[i]

In [6]:
from copy import deepcopy

In [7]:
def ga(num_resources, costs, facility_cost, population_size, num_generations, elitism_size, mutation_prob):
    population = [Individual(num_resources, costs, facility_cost) for _ in range(population_size)]
    
    if elitism_size < 0:
        elitism_size = abs(elitism_size)
    elitism = int(population_size * elitism_size)
    
    if (population_size - elitism) % 2:
        elitism += 1
    
    for _ in range(num_generations):
        # NAPOMENA: funkcija sorted() podrazumevano sortira neopadajuće. U ovom slučaju, kada želiimo da minimizu-
        # jemo cene, to je željeno ponašanje jer su najbolje jedinke upravo one koje imaju najmanje vrednosti. 
        # U primerima gde su najbolje jedinke one koje imaju najveću vrednost funkcije prilagođenosti, trebalo bi 
        # sortirati u nerastućem poretku:
        # population = sorted(population, key = lambda x: x.fitness, reverse = True)
        
        population = sorted(population, key = lambda x: x.fitness)
        new_population = deepcopy(population[:elitism])
        
        for i in range(int((population_size - elitism) / 2)):
            parent1, parent2 = tournament_selection(population, 10)
            
            child1, child2 = crossover(parent1, parent2)
            mutation(child1, mutation_prob)
            mutation(child2, mutation_prob)
            
            child1.fitness = child1.calc_fitness(costs, facility_cost)
            child2.fitness = child2.calc_fitness(costs, facility_cost)
            
            new_population.append(child1)
            new_population.append(child2)
            
        population = deepcopy(new_population)
    return min(population, key= lambda x: x.fitness)

In [8]:
import sys
sys.path.append("..")

from uflp_utils import read_instance, read_bk_instance

In [9]:
_, num_resources, costs, facility_cost = read_instance("../cas2/uflp1.txt")
num_resources, costs, facility_cost

(3,
 array([[ 1, 12,  3],
        [ 2,  7, 41],
        [19, 21,  7]]),
 array([12, 11, 13]))

In [10]:
best_individual = ga(num_resources, costs, facility_cost, 100, 100, 0.2, 0.05)
print(best_individual.code)
print(best_individual.fitness)

[True, False, False]
34


In [11]:
_, num_resources_bk, costs_bk, facility_cost_bk = read_bk_instance("../BildeKrarup/B/B1.1")

In [12]:
best_individual = ga(num_resources_bk, costs_bk, facility_cost_bk, 100, 100, 0.2, 0.05)
print(best_individual.code)
print(best_individual.fitness)

[False, False, False, False, False, False, False, False, False, False, True, False, False, True, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, False, False]
23468


## Genetski algoritam za rešavanje TSP problema

In [13]:
from tsp_utils import read_tsp

In [14]:
cost_matrix = read_tsp("../cas3/bays29.tsp")
cost_matrix

array([[  0, 107, 241, 190, 124,  80, 316,  76, 152, 157, 283, 133, 113,
        297, 228, 129, 348, 276, 188, 150,  65, 341, 184,  67, 221, 169,
        108,  45, 167],
       [107,   0, 148, 137,  88, 127, 336, 183, 134,  95, 254, 180, 101,
        234, 175, 176, 265, 199, 182,  67,  42, 278, 271, 146, 251, 105,
        191, 139,  79],
       [241, 148,   0, 374, 171, 259, 509, 317, 217, 232, 491, 312, 280,
        391, 412, 349, 422, 356, 355, 204, 182, 435, 417, 292, 424, 116,
        337, 273,  77],
       [190, 137, 374,   0, 202, 234, 222, 192, 248,  42, 117, 287,  79,
        107,  38, 121, 152,  86,  68,  70, 137, 151, 239, 135, 137, 242,
        165, 228, 205],
       [124,  88, 171, 202,   0,  61, 392, 202,  46, 160, 319, 112, 163,
        322, 240, 232, 314, 287, 238, 155,  65, 366, 300, 175, 307,  57,
        220, 121,  97],
       [ 80, 127, 259, 234,  61,   0, 386, 141,  72, 167, 351,  55, 157,
        331, 272, 226, 362, 296, 232, 164,  85, 375, 249, 147, 301, 118,
    

In [15]:
class Individual:
    def __init__(self, cost_matrix):
        self.code = list(range(len(cost_matrix)))
        random.shuffle(self.code)
        self.fitness = self.calc_fitness(cost_matrix)
        
    def calc_fitness(self, cost_matrix):
        total_cost = 0
        
        for i in range(len(self.code) - 1):
            total_cost += cost_matrix[self.code[i]][self.code[i+1]]
        total_cost += cost_matrix[self.code[-1]][self.code[0]]
        return total_cost

In [16]:
# Ruletska selekcija: pravi se mehanizam sličan ruletu, ali tako da bolje prilagođene jedinke imaju veće šanse da
# budu izabrane od onih loše prilagođenih

def roulette_selection(population):
    # NAPOMENA: pošto je potrebno minimizovati cenu puta, komanda x.fitness/sum_fitness bi zapravo favorizovala
    # LOŠIJE prilagođene jedinke. U slučajevima kada je potrebno maksimizovati vrednosti dobijene fitnes funkcijom,
    # pomenuti kod bi radio kako treba. Sa druge strane, (1/x.fitness) će favorizovati bolje jedinke. Međutim, tada
    # se verovatnoće dobijene sa (1/x.fitness)/sum_fitness ne bi sabirale na 1, pa je dobijeni vektor potrebno
    # normalizovati.
    selection_prob = np.array([(1 / x.fitness) for x in population])

    # Normalizacija
    selection_prob = selection_prob / (sum(selection_prob))
    
    parent1, parent2 = np.random.choice(population, size = 2, replace = False, p = selection_prob)
    
    return parent1, parent2

In [17]:
def crossover(parent1, parent2):
    child1 = deepcopy(parent1)
    child2 = deepcopy(parent2)
    
    # jednopoziciono ukrštanje: odaberemo nasumično jednu poziciju iz intervala: n = [0, len(parent1.code)), ili
    # n = [1, len(parent1.code) - 1) ukoliko ne želimo da dopustimo 0
    # Jedno dete onda nasleđuje prvih n bitova iz prvog, a len(parent1.code), a ostale iz drugog roditelja, dok 
    # za drugo dete važi suprotno
    n = random.randrange(len(child1.code) - 1) + 1
    k = n
    
    # Međutim, kod problema TSP može se desiti da ovom tehnikom dobijemo rešenje koje dva puta obilazi isti grad,
    # dok bar jedan grad izostaje sa spiska obiđenih. To se ne sme desiti, te je stoga potrebno nekako izmeniti 
    # logiku jednopozicionog ukrštanja. Ideja je sledeća: svako dete će naslediti prvih n gena (gradova) od po 
    # jednog roditelja. Za dobijanje vrednosti ostalih len(parent1.code) - n gradova potrebno je redom obići
    # gradove drugog roditelja i čim naiđemo na onaj koji se ne nalazi u kopiranih prvih n bitova dodajemo 
    # ga na prvu slobodnu poziciju. Analogno uradimo i sa drugim detetom.
    for i in range(len(parent2.code)):
        if parent2.code[i] not in child1.code[:n]:
            child1.code[k] = parent2.code[i]
            k += 1
    k = n
    
    for i in range(len(parent1.code)):
        if parent1.code[i] not in child2.code[:n]:
            child2.code[k] = parent1.code[i]
            k += 1
            
    return child1, child2

In [18]:
def mutation(child, mutation_prob):
    if random.random() < mutation_prob:
        i = random.randrange(len(child.code))
        j = random.randrange(len(child.code))
        
        while i == j:
            j = random.randrange(len(child.code))
    
        child.code[i], child.code[j] = child.code[j], child.code[i]

In [19]:
def ga(cost_matrix, population_size, num_generations, elitism_size, mutation_prob):
    population = [Individual(cost_matrix) for _ in range(population_size)]
    
    if elitism_size < 0:
        elitism_size = abs(elitism_size)
    elitism = int(population_size * elitism_size)
    
    if (population_size - elitism) % 2:
        elitism += 1
    
    for _ in range(num_generations):
        population = sorted(population, key = lambda x: x.fitness)
        new_population = deepcopy(population[:elitism])
        
        for i in range(int((population_size - elitism) / 2)):
            parent1, parent2 = roulette_selection(population)
            
            child1, child2 = crossover(parent1, parent2)
            mutation(child1, mutation_prob)
            mutation(child2, mutation_prob)
            
            child1.fitness = child1.calc_fitness(cost_matrix)
            child2.fitness = child2.calc_fitness(cost_matrix)
            
            new_population.append(child1)
            new_population.append(child2)
            
        population = deepcopy(new_population)
    return min(population, key= lambda x: x.fitness)

In [20]:
best_individual = ga(cost_matrix, 100, 100, 0.2, 0.05)
print(best_individual.code)
print(best_individual.fitness)

[12, 1, 28, 2, 20, 7, 26, 22, 15, 14, 13, 16, 17, 9, 19, 0, 27, 11, 8, 25, 4, 5, 23, 3, 21, 10, 6, 24, 18]
2861


### Kombinovanje genetskih algoritama sa S-metaheuristikama

Genetski algoritmi se mogu kombinovati sa S-metaheuristikama, poput simuliranog kaljenja i lokalne pretrage, kako bi se iskoristile prednosti oba pristupa. Na primer, S-metaheuristike se mogu koristiti za generisanje kvalitetnih početnih rešenja, koja će zatim činiti deo početne populacije genetskog algoritma. Druga mogućnost jeste primena S-metaheuristika na novonastale jedinke, nakon ukrštanja i mutacije, kako bi se dodatno poboljšao kvalitet dobijenih rešenja pre njihovog uključivanja u narednu generaciju. Na taj način, genetski algoritam omogućava pretragu različitih delova prostora rešenja, dok S-metaheuristike omogućavaju intenzivniju pretragu okoline pojedinačnih rešenja. Ovakvi hibridni pristupi, koji kombinuju populacionu evoluciju sa lokalnim poboljšavanjem rešenja, često se nazivaju **memetičkim algoritmima**. Genetski algoritmi se mogu koristiti i za optimizaciju parametara drugih modela ili algoritama, pri čemu svaka jedinka predstavlja jednu kombinaciju vrednosti parametara, a funkcija prilagođenosti meri kvalitet rezultata koji se tim parametrima postiže.